In [8]:
# Cell 2: Data Loading and Initial Exploration
print("📊 CELL 2: DATA LOADING AND EXPLORATION")
print("=" * 45)
print("🎯 Objective: Load same data as final_baseline_models")
print("📋 Focus: Understand data structure and target distribution")
print()

# Load the data from the same source as final_baseline_models
df = pd.read_excel('MS_Capstone_Final_Quality_Check_Summary.xlsx', sheet_name='clean_data')

print("✅ Data loaded successfully")
print(f"📊 Dataset shape: {df.shape}")
print(f"   - Rows (observations): {df.shape[0]}")
print(f"   - Columns (features): {df.shape[1]}")
print()

# Display basic information about the dataset
print("📋 DATASET OVERVIEW:")
print("=" * 25)
print("Column names and types:")
print(df.dtypes)
print()

# Check the target variable (same as baseline models)
target_col = 'Movement Label (1,0)'
if target_col in df.columns:
    print(f"🎯 TARGET VARIABLE: {target_col}")
    print("Target distribution:")
    target_counts = df[target_col].value_counts()
    print(target_counts)
    print(f"   - Class balance: {target_counts[1]}/{target_counts[0]} (Up/Down)")
    print(f"   - Percentage Up: {target_counts[1]/len(df)*100:.1f}%")
    print(f"   - Percentage Down: {target_counts[0]/len(df)*100:.1f}%")
else:
    print("❌ Target variable not found. Available columns:")
    print(df.columns.tolist())

print()
print("🔍 Ready for Cell 3: Feature Selection...")

📊 CELL 2: DATA LOADING AND EXPLORATION
🎯 Objective: Load same data as final_baseline_models
📋 Focus: Understand data structure and target distribution

✅ Data loaded successfully
📊 Dataset shape: (585, 32)
   - Rows (observations): 585
   - Columns (features): 32

📋 DATASET OVERVIEW:
Column names and types:
Company                                object
Ticker                                 object
AV_Ticker.BSE                          object
Sector                                 object
Cap Category                           object
Period                                 object
Sales                                 float64
Expenses                              float64
Operating Profit                      float64
OPM %                                 float64
Other Income                          float64
Interest                              float64
Depreciation                          float64
Profit before tax                     float64
Tax %                                 float64
N

In [9]:
# Cell 3: Feature Selection and Data Preprocessing
print("🔧 CELL 3: FEATURE SELECTION AND PREPROCESSING")
print("=" * 50)
print("🎯 Objective: Select same predictors as final_baseline_models")
print("📋 Focus: Handle NaN values and categorical variables")
print()

# First, check for NaN values in target variable
print("🎯 TARGET VARIABLE ANALYSIS:")
print("=" * 30)
target_col = 'Movement Label (1,0)'
print(f"Target column: {target_col}")
print(f"Total NaN in target: {df[target_col].isnull().sum()}")
print(f"Target data type: {df[target_col].dtype}")

# Remove rows with NaN target values
initial_rows = len(df)
df_clean = df.dropna(subset=[target_col]).copy()
rows_removed = initial_rows - len(df_clean)

print(f"Rows before cleaning: {initial_rows}")
print(f"Rows after removing NaN targets: {len(df_clean)}")
print(f"Rows removed: {rows_removed}")
print()

# Define feature columns (same as baseline models - financial indicators only)
print("📊 FEATURE SELECTION:")
print("=" * 25)

# Exclude non-predictive columns (identifiers, dates, target)
exclude_cols = [
    'Company', 'Ticker', 'AV_Ticker.BSE', 'Period',  # Identifiers
    'Result Date', '_T_date', '_T1_date', '_T2_date', '_T3_date',  # Dates
    'Movement Label (1,0)', # Target variable
    # FUTURE INFORMATION FEATURES (DATA LEAKAGE)
    'Closing Price (T)',           # Price on result day
    'Closing Price (T+1)',         # Price 1 day after result
    'Closing Price (T+2)',         # Price 2 days after result  
    'Closing Price (T+3)',         # Price 3 days after result
    '3-Day Avg Price Post Result', # Average price AFTER result
    '3-Day Return (%)' 
]

# Identify numerical financial indicators
numerical_features = []
categorical_features = []

for col in df_clean.columns:
    if col not in exclude_cols:
        if df_clean[col].dtype in ['int64', 'float64']:
            numerical_features.append(col)
        elif df_clean[col].dtype == 'object':
            categorical_features.append(col)

print(f"📈 Numerical features identified: {len(numerical_features)}")
print("Numerical features:", numerical_features[:10], "..." if len(numerical_features) > 10 else "")
print()
print(f"📝 Categorical features identified: {len(categorical_features)}")
print("Categorical features:", categorical_features)
print()

# Check for missing values in features
print("🔍 MISSING VALUES ANALYSIS:")
print("=" * 30)
all_features = numerical_features + categorical_features
missing_summary = df_clean[all_features].isnull().sum()
features_with_missing = missing_summary[missing_summary > 0]

if len(features_with_missing) > 0:
    print("Features with missing values:")
    for feature, count in features_with_missing.items():
        percentage = (count / len(df_clean)) * 100
        print(f"   {feature}: {count} ({percentage:.1f}%)")
else:
    print("✅ No missing values in feature columns")

print()
print("🎯 Ready for Cell 4: Data Preparation Pipeline...")

🔧 CELL 3: FEATURE SELECTION AND PREPROCESSING
🎯 Objective: Select same predictors as final_baseline_models
📋 Focus: Handle NaN values and categorical variables

🎯 TARGET VARIABLE ANALYSIS:
Target column: Movement Label (1,0)
Total NaN in target: 1
Target data type: float64
Rows before cleaning: 585
Rows after removing NaN targets: 584
Rows removed: 1

📊 FEATURE SELECTION:
📈 Numerical features identified: 14
Numerical features: ['Sales', 'Expenses', 'Operating Profit', 'OPM %', 'Other Income', 'Interest', 'Depreciation', 'Profit before tax', 'Tax %', 'Net Profit'] ...

📝 Categorical features identified: 2
Categorical features: ['Sector', 'Cap Category']

🔍 MISSING VALUES ANALYSIS:
Features with missing values:
   Sales: 117 (20.0%)
   Operating Profit: 117 (20.0%)
   OPM %: 117 (20.0%)
   Revenue: 467 (80.0%)
   Financing Profit: 467 (80.0%)
   Financing Margin %: 467 (80.0%)

🎯 Ready for Cell 4: Data Preparation Pipeline...


In [15]:
# Cell 4: Data Preparation Pipeline (REVISED)
print("🔧 CELL 4: DATA PREPARATION PIPELINE (REVISED)")
print("=" * 50)
print("🎯 Objective: Create robust preprocessing pipeline")
print("📋 Focus: Keep all features to preserve sector representation")
print()

# First, let's analyze the sector distribution and missing values by sector
print("🏦 SECTOR ANALYSIS WITH MISSING VALUES:")
print("=" * 40)

# Check sector distribution
sector_dist = df_clean['Sector'].value_counts()
print("Sector distribution:")
print(sector_dist)
print()

# Check which sectors have the missing financial data
missing_features = ['Revenue', 'Financing Profit', 'Financing Margin %']
print("Missing values analysis by sector:")
for feature in missing_features:
    print(f"\
{feature}:")
    missing_by_sector = df_clean.groupby('Sector')[feature].apply(lambda x: x.isnull().sum())
    total_by_sector = df_clean.groupby('Sector')[feature].apply(lambda x: len(x))
    missing_pct_by_sector = (missing_by_sector / total_by_sector * 100).round(1)
    
    for sector in sector_dist.index:
        missing_count = missing_by_sector.get(sector, 0)
        total_count = total_by_sector.get(sector, 0)
        missing_pct = missing_pct_by_sector.get(sector, 0)
        print(f"   {sector}: {missing_count}/{total_count} ({missing_pct}%) missing")

print()

# REVISED STRATEGY: Keep all features but use sector-aware imputation
print("📊 REVISED FEATURE SELECTION STRATEGY:")
print("=" * 40)
print("✅ Keeping ALL features to preserve sector representation")
print("🔧 Using sector-aware imputation for missing values")
print()

# Keep all features
all_features = numerical_features + categorical_features
final_numerical_features = numerical_features.copy()
final_categorical_features = categorical_features.copy()

print(f"📈 Total numerical features: {len(final_numerical_features)}")
print(f"📝 Total categorical features: {len(final_categorical_features)}")
print("Categorical features:", final_categorical_features)
print()

# Prepare the final dataset
print("🔧 PREPARING FINAL DATASET:")
print("=" * 30)

# Create feature matrix X and target vector y
X = df_clean[all_features].copy()
y = df_clean[target_col].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print()

# SECTOR-AWARE IMPUTATION STRATEGY
print("🔧 SECTOR-AWARE MISSING VALUE HANDLING:")
print("=" * 40)

# For numerical features: use sector-specific median, fallback to overall median
for feature in final_numerical_features:
    if X[feature].isnull().sum() > 0:
        print(f"Imputing {feature}:")
        
        # Calculate sector-specific medians
        sector_medians = df_clean.groupby('Sector')[feature].median()
        overall_median = df_clean[feature].median()
        
        # Apply sector-specific imputation
        for sector in df_clean['Sector'].unique():
            sector_mask = (X['Sector'] == sector) & X[feature].isnull()
            if sector_mask.sum() > 0:
                # Use sector median if available, otherwise overall median
                impute_value = sector_medians.get(sector, overall_median)
                if pd.isna(impute_value):
                    impute_value = overall_median
                
                X.loc[sector_mask, feature] = impute_value
                print(f"   {sector}: {sector_mask.sum()} values imputed with {impute_value:.2f}")

# For categorical features: use mode imputation
for feature in final_categorical_features:
    if X[feature].isnull().sum() > 0:
        mode_value = X[feature].mode()[0]
        X[feature] = X[feature].fillna(mode_value)
        print(f"   Imputed {feature} with mode: {mode_value}")

# Verify no missing values remain
remaining_missing = X.isnull().sum().sum()
print(f"\
Remaining missing values: {remaining_missing}")

if remaining_missing == 0:
    print("✅ All missing values handled successfully")
    print("✅ All sectors preserved with complete feature representation")
else:
    print("❌ Some missing values remain - need additional handling")

print()
print("🎯 Ready for Cell 5: Pipeline Setup with Categorical Encoding...")

🔧 CELL 4: DATA PREPARATION PIPELINE (REVISED)
🎯 Objective: Create robust preprocessing pipeline
📋 Focus: Keep all features to preserve sector representation

🏦 SECTOR ANALYSIS WITH MISSING VALUES:
Sector distribution:
Sector
IT / Technology    117
BFSI               117
FMCG               117
Pharma             117
Auto               116
Name: count, dtype: int64

Missing values analysis by sector:
Revenue:
   IT / Technology: 117/117 (100.0%) missing
   BFSI: 0/117 (0.0%) missing
   FMCG: 117/117 (100.0%) missing
   Pharma: 117/117 (100.0%) missing
   Auto: 116/116 (100.0%) missing
Financing Profit:
   IT / Technology: 117/117 (100.0%) missing
   BFSI: 0/117 (0.0%) missing
   FMCG: 117/117 (100.0%) missing
   Pharma: 117/117 (100.0%) missing
   Auto: 116/116 (100.0%) missing
Financing Margin %:
   IT / Technology: 117/117 (100.0%) missing
   BFSI: 0/117 (0.0%) missing
   FMCG: 117/117 (100.0%) missing
   Pharma: 117/117 (100.0%) missing
   Auto: 116/116 (100.0%) missing

📊 REVISED FEA

In [16]:

# RQ3_Sectorwise_Methodology.py
# RQ3: How does the predictive performance of structured financial indicators vary across different sectors in the Indian equity market?
# Methodology: The test dataset will be stratified by sector, and model performance will be evaluated separately for each group.

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
from sklearn.inspection import permutation_importance

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import shap
#shap.logger.setLevel('ERROR')

# =========================
# CONFIG
# =========================
DATA_PATH = 'MS_Capstone_Final_Quality_Check_Summary.xlsx'
SECTOR_COL_CANDIDATES = ['Sector']
TARGET_COL_CANDIDATES = ['Movement Label (1,0)']
ID_LIKE_SUBSTR = ['id', 'ID', 'ticker', 'symbol', 'isin', 'name', 'Code', 'code']
RANDOM_STATE = 42
N_JOBS = -1
N_ITER = 30   # random search iterations per model
CV_FOLDS = 5
TEST_SIZE = 0.2   # per-sector test size
VAL_NOTE = 'Per-sector split: train/val via CV; test is holdout stratified within sector.'
OUT_DIR = 'rq3_outputs'

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================

def find_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    # try case-insensitive
    lower_map = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None


def infer_features(df, sector_col, target_col):
    # numeric features only, drop sector/target and any ID-like columns
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    features = [c for c in num_cols if c not in [target_col]]
    # remove obvious ids in numeric space if any (rare)
    features = [c for c in features if not any(s in c for s in ID_LIKE_SUBSTR)]

    # remove non-numeric that slipped in
    # (handled by dtype already)

    # ensure not empty
    if len(features) == 0:
        raise ValueError('No numeric feature columns found.')
    return features


def build_models_and_params():
    models = {
        'LogisticRegression': Pipeline([
            ('scaler', StandardScaler(with_mean=True)),
            ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
        ]),
        'RandomForest': Pipeline([
            ('scaler', StandardScaler(with_mean=False)),
            ('clf', RandomForestClassifier(random_state=RANDOM_STATE))
        ]),
        'XGBoost': Pipeline([
            ('scaler', StandardScaler(with_mean=False)),
            ('clf', XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=N_JOBS))
        ])
    }

    param_grids = {
        'LogisticRegression': {
            'clf__C': np.logspace(-3, 2, 20),
            'clf__penalty': ['l2'],
            'clf__solver': ['lbfgs', 'liblinear']
        },
        'RandomForest': {
            'clf__n_estimators': [200, 300, 500, 800],
            'clf__max_depth': [None, 4, 6, 8, 12],
            'clf__min_samples_split': [2, 5, 10],
            'clf__min_samples_leaf': [1, 2, 4],
            'clf__max_features': ['sqrt', 'log2', None]
        },
        'XGBoost': {
            'clf__n_estimators': [200, 400, 600, 800],
            'clf__max_depth': [3, 4, 5, 6],
            'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'clf__subsample': [0.7, 0.8, 1.0],
            'clf__colsample_bytree': [0.6, 0.8, 1.0],
            'clf__reg_lambda': [0.0, 0.5, 1.0, 2.0],
            'clf__reg_alpha': [0.0, 0.1, 0.5]
        }
    }

    return models, param_grids


def fit_with_cv(model, params, X_train, y_train, scoring='f1'):
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=N_ITER,
        scoring=scoring,
        cv=cv,
        verbose=0,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    return search


def predict_scores(estimator, X):
    # get probabilities or decision scores if available
    if hasattr(estimator, 'predict_proba'):
        try:
            return estimator.predict_proba(X)[:, 1]
        except Exception:
            pass
    if hasattr(estimator, 'decision_function'):
        try:
            return estimator.decision_function(X)
        except Exception:
            pass
    return estimator.predict(X)


def evaluate(estimator, X, y_true):
    y_pred = estimator.predict(X)
    y_score = predict_scores(estimator, X)
    out = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0)
    }
    try:
        out['AUC'] = roc_auc_score(y_true, y_score)
    except Exception:
        out['AUC'] = np.nan
    return out


def compute_shap_for_pipeline(fitted_pipe, X_train, X_test):
    clf = fitted_pipe.named_steps['clf']
    model_name = clf.__class__.__name__

    # scaled data for linear, raw for trees
    if 'scaler' in fitted_pipe.named_steps:
        X_train_scaled = fitted_pipe.named_steps['scaler'].transform(X_train)
        X_test_scaled = fitted_pipe.named_steps['scaler'].transform(X_test)
    else:
        X_train_scaled = X_train
        X_test_scaled = X_test

    if 'LogisticRegression' in model_name:
        explainer = shap.LinearExplainer(clf, X_train_scaled)
        shap_vals = explainer.shap_values(X_test_scaled)
        data_for_plot = X_test_scaled
    elif 'RandomForest' in model_name or 'XGBClassifier' in model_name:
        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(X_test)
        data_for_plot = X_test
    else:
        explainer = shap.Explainer(clf, X_train_scaled)
        shap_vals = explainer(X_test_scaled)
        data_for_plot = X_test_scaled

    if isinstance(shap_vals, list) and len(shap_vals) == 2:
        shap_vals = shap_vals[1]

    return shap_vals, data_for_plot


def plot_save_bar(df, x, y, hue, title, fname):
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df, x=x, y=y, hue=hue, edgecolor='black')
    plt.ylim(0, 1)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, fname)
    plt.savefig(out_path, dpi=200)
    plt.show()
    print('Saved figure: ' + out_path)


def plot_save_heatmap(pivot_df, title, fname):
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Sector')
    plt.ylabel('Model')
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, fname)
    plt.savefig(out_path, dpi=200)
    plt.show()
    print('Saved figure: ' + out_path)

# =========================
# MAIN FLOW
# =========================

def main():
    # Try to read first sheet by default
    df = df_clean
    #df = pd.read_excel('MS_Capstone_Final_Quality_Check_Summary.xlsx', sheet_name='clean_data')
    #print('Loaded rows: ' + str(len(df)))

    # 2) Identify sector and target columns
    sector_col = find_column(df, SECTOR_COL_CANDIDATES)
    target_col = find_column(df, TARGET_COL_CANDIDATES)

    if sector_col is None:
        raise ValueError('Could not find sector column. Please rename or update candidates.')
    if target_col is None:
        raise ValueError('Could not find target column. Please rename or update candidates.')

    # Drop rows with NA in target or sector
    df = df.dropna(subset=[sector_col, target_col]).copy()
    df[sector_col] = df[sector_col].astype(str).str.strip()
    df[target_col] = df[target_col].astype(int)

    # 3) Build feature list from numeric columns
    features = infer_features(df, sector_col, target_col)
    print('Using ' + str(len(features)) + ' numeric features')

    # 4) Split TEST set stratified by sector
    # We create a per-sector split so each sector gets its own holdout
    results = []
    best_models = {}
    shap_outputs = {}
    perm_outputs = {}

    sectors = sorted(df[sector_col].unique().tolist())
    print('Sectors detected: ' + ', '.join(sectors))

    for sec in sectors:
        df_sec = df[df[sector_col] == sec].copy()
        # Skip sectors with insufficient classes
        if df_sec[target_col].nunique() < 2:
            print('Skipping sector ' + sec + ' due to single-class target')
            continue

        X = df_sec[features].copy()
        y = df_sec[target_col].copy()

        # Per-sector stratified split
        # If small counts break stratification, fallback without stratify
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
            )
        except ValueError:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
            )

        # 5) Build models and tune with CV on training only
        models, param_grids = build_models_and_params()
        tuned_models = {}
        for mname, pipe in models.items():
            print('Tuning ' + mname + ' for sector ' + sec)
            search = fit_with_cv(pipe, param_grids[mname], X_train, y_train, scoring='f1')
            tuned_models[mname] = search.best_estimator_

        best_models[sec] = tuned_models

        # 6) Evaluate tuned models on sector test set
        for mname, est in tuned_models.items():
            mets = evaluate(est, X_test, y_test)
            entry = {
                'Sector': sec,
                'Model': mname,
                'Accuracy': mets['Accuracy'],
                'Precision': mets['Precision'],
                'Recall': mets['Recall'],
                'F1_Score': mets['F1_Score'],
                'AUC': mets.get('AUC', np.nan)
            }
            results.append(entry)

        # 7) SHAP and Permutation importance for each model
        shap_outputs[sec] = {}
        perm_outputs[sec] = {}
        for mname, est in tuned_models.items():
            try:
                shap_vals, data_for_plot = compute_shap_for_pipeline(est, X_train, X_test)
                shap_outputs[sec][mname] = {
                    'values': shap_vals,
                    'X_plot': data_for_plot,
                    'features': features
                }

                # Save SHAP importance bar (top 10)
                imp_df = pd.DataFrame({
                    'Feature': features,
                    'Importance': np.abs(shap_vals).mean(axis=0)
                }).sort_values('Importance', ascending=False).head(10)
                plt.figure(figsize=(10, 7))
                colors = plt.cm.Blues(np.linspace(0.45, 0.85, len(imp_df)))
                plt.barh(range(len(imp_df)), imp_df['Importance'], color=colors, edgecolor='black')
                plt.yticks(range(len(imp_df)), imp_df['Feature'])
                plt.gca().invert_yaxis()
                plt.title(sec + ' - ' + mname + ' Top 10 SHAP Importance')
                plt.xlabel('Mean |SHAP Value|')
                plt.tight_layout()
                fig_path = os.path.join(OUT_DIR, 'SHAP_Top10_' + sec.replace(' ', '_') + '_' + mname + '.png')
                plt.savefig(fig_path, dpi=200)
                plt.show()
                print('Saved figure: ' + fig_path)

                # SHAP summary plot (top 15)
                plt.figure(figsize=(11, 7))
                shap.summary_plot(shap_vals, data_for_plot, feature_names=features, show=False, max_display=15)
                plt.title(sec + ' - ' + mname + ' SHAP Summary', fontsize=13)
                plt.tight_layout()
                fig_path = os.path.join(OUT_DIR, 'SHAP_Summary_' + sec.replace(' ', '_') + '_' + mname + '.png')
                plt.savefig(fig_path, dpi=200, bbox_inches='tight')
                plt.show()
                print('Saved figure: ' + fig_path)
            except Exception as e:
                print('SHAP failed for sector ' + sec + ' - ' + mname + ': ' + str(e))

            try:
                perm_imp = permutation_importance(est, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='f1', n_jobs=N_JOBS)
                perm_df = pd.DataFrame({
                    'Feature': features,
                    'Importance': perm_imp.importances_mean,
                    'Std': perm_imp.importances_std
                }).sort_values('Importance', ascending=False).head(10)
                perm_outputs[sec][mname] = perm_df
                plt.figure(figsize=(10, 7))
                colors = plt.cm.Greens(np.linspace(0.45, 0.85, len(perm_df)))
                plt.barh(range(len(perm_df)), perm_df['Importance'], color=colors, edgecolor='black')
                plt.yticks(range(len(perm_df)), perm_df['Feature'])
                plt.gca().invert_yaxis()
                plt.title(sec + ' - ' + mname + ' Permutation Importance (Top 10)')
                plt.xlabel('Mean Decrease in Score')
                plt.tight_layout()
                fig_path = os.path.join(OUT_DIR, 'PermImp_Top10_' + sec.replace(' ', '_') + '_' + mname + '.png')
                plt.savefig(fig_path, dpi=200)
                plt.show()
                print('Saved figure: ' + fig_path)
            except Exception as e:
                print('Permutation Importance failed for sector ' + sec + ' - ' + mname + ': ' + str(e))

    # Aggregate results
    results_df = pd.DataFrame(results)
    results_path = os.path.join(OUT_DIR, 'RQ3_Sector_Results.xlsx')
    if len(results_df) > 0:
        # Save metrics
        with pd.ExcelWriter(results_path) as writer:
            results_df.to_excel(writer, index=False, sheet_name='All_Results')
        print('Saved results to: ' + results_path)

        # Visualizations across sectors
        for metric in ['F1_Score', 'Accuracy', 'Precision', 'Recall', 'AUC']:
            plot_save_bar(results_df, x='Sector', y=metric, hue='Model', title=metric + ' by Sector and Model', fname='Bar_' + metric + '.png')

        # Heatmaps: pivot Model x Sector
        for metric in ['F1_Score', 'Accuracy']:
            pivot_df = results_df.pivot_table(index='Model', columns='Sector', values=metric, aggfunc='mean')
            plot_save_heatmap(pivot_df, title=metric + ' Heatmap by Sector and Model', fname='Heatmap_' + metric + '.png')

    else:
        print('No results to summarize; check data and columns.')

    # Hypothesis Reporting (printed)
    print('RQ3 Outcome: Sector-wise predictability')
    print('- We compared F1 and Accuracy (plus other metrics) across sectors on holdout test sets stratified per sector.')
    print('- Inspect the Excel and figures in ' + OUT_DIR + ' to see which sectors show higher/lower predictive performance.')

if __name__ == '__main__':
    print('Running RQ3 Sectorwise Methodology pipeline...')
    main()


Running RQ3 Sectorwise Methodology pipeline...
Using 20 numeric features
Sectors detected: Auto, BFSI, FMCG, IT / Technology, Pharma
Tuning LogisticRegression for sector Auto


ValueError: 
All the 150 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "A:\anaconda\envs\capstone\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "A:\anaconda\envs\capstone\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "A:\anaconda\envs\capstone\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "A:\anaconda\envs\capstone\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *a